# Experiment 4: Repairing a Lost Share

When a participant loses their secret share (hardware failure, corrupted
backup), the remaining participants can reconstruct it without any single
helper learning the group secret. This uses additive splitting: each helper's
Lagrange-weighted contribution is broken into random pieces distributed among
the helpers.


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Aggregator, Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash
from frost.lagrange import lagrange_coefficient

# DKG setup (2-of-3)
threshold = 2
n = 3
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]

for p in participants:
    p.init_keygen()
for p in participants:
    p.generate_shares()
for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)
for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants if other.index != p.index
    )
    p.derive_public_key(other_commitments)
for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

public_key = participants[0].public_key
print(f"DKG complete. Public key x = {public_key.x}")
print(f"P3 aggregate share: {participants[2].aggregate_share}")


## Simulate Share Loss

Save P3's share, then set it to None to simulate loss.


In [ ]:
original_share = participants[2].aggregate_share
participants[2].aggregate_share = None
print(f"P3 share saved: {original_share}")
print(f"P3 share now: {participants[2].aggregate_share}")


## Repair Protocol

Helpers P1 and P2 collaborate to reconstruct P3's share. Each helper:
1. Generates repair shares (random additive splits of their weighted contribution)
2. Exchanges repair shares with the other helper
3. Aggregates received repair shares

Then P3 sums the aggregate repair shares to recover their original share.


In [ ]:
# Step 1: Helpers generate repair shares
# Args: (other_helper_indexes, target_index)
participants[0].generate_repair_shares((2,), 3)  # P1 helps repair P3, other helper is P2
participants[1].generate_repair_shares((1,), 3)  # P2 helps repair P3, other helper is P1

print("Repair shares generated.")
print(f"P1 repair participants: {participants[0].repair_participants}")
print(f"P2 repair participants: {participants[1].repair_participants}")

# Step 2: Exchange repair shares
p1_from_p2 = participants[1].get_repair_share(1)  # P2's share designated for P1
p2_from_p1 = participants[0].get_repair_share(2)  # P1's share designated for P2
print(f"P1 receives from P2: {p1_from_p2}")
print(f"P2 receives from P1: {p2_from_p1}")

# Step 3: Aggregate
participants[0].aggregate_repair_shares((p1_from_p2,))
participants[1].aggregate_repair_shares((p2_from_p1,))
print(f"P1 aggregate repair share: {participants[0].aggregate_repair_share}")
print(f"P2 aggregate repair share: {participants[1].aggregate_repair_share}")

In [ ]:
# Step 4: P3 reconstructs
participants[2].repair_share((
    participants[0].aggregate_repair_share,
    participants[1].aggregate_repair_share,
))

print(f"Original share: {original_share}")
print(f"Recovered share: {participants[2].aggregate_share}")
print(f"Match: {original_share == participants[2].aggregate_share}")


## Sign with the Repaired Share

The ultimate test: can P3 participate in signing with their recovered share?


In [ ]:
message = b"Signed after repair"
signers = [participants[0], participants[2]]  # P1 and repaired P3
signer_indexes = tuple(p.index for p in signers)

for signer in signers:
    signer.generate_nonce_pair()

all_nonce_pairs = [(Point(), Point())] * n
for signer in signers:
    all_nonce_pairs[signer.index - 1] = signer.nonce_commitment_pair
nonce_commitment_pairs = tuple(all_nonce_pairs)

shares = []
for signer in signers:
    z_i = signer.sign(message, nonce_commitment_pairs, signer_indexes)
    shares.append(z_i)

agg = Aggregator(
    public_key, message, nonce_commitment_pairs, signer_indexes,
)
sig_hex = agg.signature(tuple(shares))

# BIP340 verification (must use even-y public key per lift_x convention)
sig_bytes = bytes.fromhex(sig_hex)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")
R = Point.lift_x(R_x)
pk = Point.lift_x(public_key.x)
e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + pk.to_bytes_xonly() + message)
e = int.from_bytes(e_bytes, "big") % Q
valid = s * G == R + (e * pk)
print(f"Signature valid with repaired share: {valid}")

## Why It Works: Lagrange Weights

The repair mechanism is Lagrange interpolation in disguise. Each helper
contributes λᵢ·sᵢ evaluated at the target index. The sum of these weighted
contributions equals f(target), the polynomial evaluated at the lost
participant's index, which is exactly their original aggregate share.


In [ ]:
# The Lagrange weights used in repair
helpers = (1, 2)
target = 3

for h in helpers:
    lam = lagrange_coefficient(helpers, h, target)
    print(f"  \u03bb_{h}(target={target}) = {int(lam)}")

# Manual reconstruction: \u2211 \u03bb\u1d62\u00b7s\u1d62 at target index
manual = Scalar(0)
for h in helpers:
    lam = lagrange_coefficient(helpers, h, target)
    s_h = Scalar(participants[h - 1].aggregate_share)
    manual = manual + lam * s_h

print(f"\nManual interpolation: {int(manual)}")
print(f"Recovered share:     {participants[2].aggregate_share}")
print(f"Match: {int(manual) == participants[2].aggregate_share}")


## What If Only 1 Helper Participates?

With threshold t=2, we need at least t=2 helpers for correct interpolation.
A single helper can generate their portion, but the math won't reconstruct
the correct share: one Lagrange weight alone can't recover the target's
polynomial evaluation.


In [ ]:
# With only P1 helping, the Lagrange coefficient for a 1-element set
# is just 1. So P1's "repair contribution" is just their own share,
# not the target's share. The reconstruction would give the wrong value.

solo_helpers = (1,)
target = 3
lam_solo = lagrange_coefficient(solo_helpers, 1, target)
print(f"Single-helper Lagrange coefficient: {int(lam_solo)} (always 1 for a 1-element set)")
print(f"This means the 'repair' would just be P1's own share, not P3's.")
print(f"\nRepair requires at least t={threshold} helpers, just like secret reconstruction")
print(f"requires at least t shares.")

## Results

The repair protocol recovers the exact original share. The repaired participant
can immediately sign, producing valid BIP340 signatures. No new DKG is needed,
and the group public key is unchanged.

The security property: no single helper learns the group secret during repair.
Each helper only sees random additive pieces of other helpers' weighted
contributions, not the contributions themselves.

## Next Steps

- This experiment verifies the same invariant as
  `test_properties.py::test_share_repair_produces_valid_share`.
- Experiment 5 shows that the same mechanism handles enrollment (new
  participants).
- See Guide Chapter 8 (Advanced Operations) for the full repair protocol.
